# 🔵 LAB D8 – ML No Supervisado: Clustering con K-Means
## Fundamentos de Gestión de Datos | TECSUP 2026-I
**Docente:** Mg. Pilar Rocío Sayán Mejía  
**Semana:** 8 | **Unidad:** II — Python para Datos + Limpieza + ML Básico


## 🎯 Capacidad a Lograr
Aplicar K-Means para segmentar registros del dataset (clientes/productos), seleccionar k óptimo con el método del codo, interpretar el perfil de cada grupo y persistir los resultados en SQLite.


---
## ⚙️ Paso 1: Carga y preparación de datos para clustering


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print("✅ Librerías listas | K-Means con scikit-learn")


In [ ]:
# Crear BD de ejemplo y cargar datos
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE clientes_limpios (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT, edad INTEGER, ciudad TEXT,
    ingreso_mensual REAL
);
CREATE TABLE ventas_limpias (
    id_venta INTEGER PRIMARY KEY, id_cliente INTEGER,
    cantidad INTEGER, monto_total REAL
);
""")

import random; random.seed(42)
n = 150
# Crear 3 perfiles naturales de clientes
perfiles = [
    (20,35, 1200,2500, 1,3, 100,400),   # jóvenes, bajo ingreso, compras pequeñas
    (35,50, 4000,7000, 4,8, 800,2000),  # adultos, alto ingreso, compras grandes
    (25,45, 2500,4000, 2,5, 400,900),   # rango medio
]
clientes, ventas = [], []
id_v = 1
for i in range(1, n+1):
    p = perfiles[i % 3]
    edad = random.randint(p[0], p[1])
    ingreso = round(random.uniform(p[2], p[3]), 1)
    clientes.append((i, f"Cliente_{i}", edad,
                     random.choice(['Lima','Arequipa','Cusco']), ingreso))
    nv = random.randint(2, 6)
    for _ in range(nv):
        cant = random.randint(p[4], p[5])
        monto = round(cant * random.uniform(p[6]//cant, p[7]//cant), 2)
        ventas.append((id_v, i, cant, monto))
        id_v += 1

conn.executemany("INSERT INTO clientes_limpios VALUES (?,?,?,?,?)", clientes)
conn.executemany("INSERT INTO ventas_limpias VALUES (?,?,?,?)", ventas)
conn.commit()

# Construir tabla de métricas por cliente (RFM simplificado)
df = pd.read_sql_query("""
    SELECT c.id_cliente, c.edad, c.ingreso_mensual,
           COUNT(v.id_venta)      AS frecuencia,
           SUM(v.monto_total)     AS gasto_total,
           AVG(v.monto_total)     AS ticket_promedio
    FROM clientes_limpios c
    JOIN ventas_limpias v ON c.id_cliente = v.id_cliente
    GROUP BY c.id_cliente
""", conn)

print(f"Dataset para clustering: {df.shape}")
print(df.describe().round(1))

# Escalar variables (K-Means es sensible a la escala)
features = ['edad','ingreso_mensual','frecuencia','gasto_total','ticket_promedio']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
print("\n✅ Variables escaladas con StandardScaler")


## 📐 Paso 2: Selección del número óptimo de clusters (Método del Codo)


In [ ]:
# PASO 2: Elbow Method + Silhouette Score
print("=== SELECCIÓN DE k ÓPTIMO ===\n")

inercias   = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))
    print(f"k={k}  Inercia={km.inertia_:,.0f}  Silhouette={silhouette_score(X_scaled,km.labels_):.3f}")

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,5))

ax1.plot(K_range, inercias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Número de Clusters (k)'); ax1.set_ylabel('Inercia (WCSS)')
ax1.set_title('📐 Método del Codo')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouettes, 'rs-', linewidth=2, markersize=8)
ax2.set_xlabel('Número de Clusters (k)'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('📊 Silhouette Score por k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/elbow_silhouette.png', dpi=100)
plt.show()

k_optimo = 3
print(f"\n✅ k elegido = {k_optimo} (punto de inflexión + mayor silhouette)")


## 🤖 Paso 3: Entrenamiento del modelo K-Means


In [ ]:
# PASO 3: Entrenar K-Means con k óptimo
print(f"=== K-MEANS con k={k_optimo} ===\n")

kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
kmeans.fit(X_scaled)

df['cluster'] = kmeans.labels_
df['cluster_nombre'] = df['cluster'].astype(str)  # se renombra en paso 4

print(f"Inercia final: {kmeans.inertia_:,.1f}")
print(f"Silhouette Score: {silhouette_score(X_scaled, kmeans.labels_):.4f}")
print("\nDistribución por cluster:")
print(df['cluster'].value_counts().sort_index().to_string())


## 👥 Paso 4: Análisis del perfil de cada cluster


In [ ]:
# PASO 4: Perfiles de cluster
print("=== PERFILES DE CLUSTER ===\n")

perfil = df.groupby('cluster')[features].agg(['mean','std']).round(1)
print(perfil)

# Estadísticas claras por cluster
resumen = df.groupby('cluster').agg(
    n_clientes=('id_cliente','count'),
    edad_promedio=('edad','mean'),
    ingreso_promedio=('ingreso_mensual','mean'),
    frecuencia_promedio=('frecuencia','mean'),
    gasto_total_promedio=('gasto_total','mean'),
    ticket_promedio=('ticket_promedio','mean')
).round(1)

print("\n📋 Resumen ejecutivo por cluster:")
print(resumen.to_string())

# Nombrar clusters según perfil
nombres_cluster = {}
for c in range(k_optimo):
    row = resumen.loc[c]
    if row['gasto_total_promedio'] > resumen['gasto_total_promedio'].mean()*1.3:
        nombres_cluster[c] = "🥇 Clientes Premium (alto valor)"
    elif row['gasto_total_promedio'] < resumen['gasto_total_promedio'].mean()*0.7:
        nombres_cluster[c] = "🎯 Clientes en Desarrollo (potencial)"
    else:
        nombres_cluster[c] = "⭐ Clientes Regulares (base)"

print("\n🏷️  Nombres asignados:")
for k,v in nombres_cluster.items():
    print(f"  Cluster {k}: {v}")
df['cluster_nombre'] = df['cluster'].map(nombres_cluster)


## 📊 Paso 5: Visualización de los clusters


In [ ]:
# PASO 5: Visualizaciones
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

# 1. Scatter: gasto_total vs frecuencia
for cluster_id in sorted(df['cluster'].unique()):
    mask = df['cluster']==cluster_id
    axes[0].scatter(df[mask]['frecuencia'], df[mask]['gasto_total'],
                    c=colors[cluster_id], label=f"C{cluster_id}", alpha=0.7, s=50, edgecolors='white')
axes[0].set_xlabel('Frecuencia de compra'); axes[0].set_ylabel('Gasto Total (S/)')
axes[0].set_title('Frecuencia vs. Gasto Total'); axes[0].legend()

# 2. PCA 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
for cluster_id in sorted(df['cluster'].unique()):
    mask = df['cluster']==cluster_id
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1],
                    c=colors[cluster_id], label=f"C{cluster_id}", alpha=0.7, s=50)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('Clusters en 2D (PCA)'); axes[1].legend()

# 3. Barras por cluster
counts = df['cluster'].value_counts().sort_index()
bars = axes[2].bar(counts.index, counts.values,
                   color=colors[:len(counts)], edgecolor='black', alpha=0.85)
for bar, n in zip(bars, counts.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 str(n), ha='center', fontweight='bold')
axes[2].set_xlabel('Cluster'); axes[2].set_ylabel('Cantidad de clientes')
axes[2].set_title('Distribución por Cluster')

plt.tight_layout()
plt.savefig('/tmp/clusters_visualizacion.png', dpi=100)
plt.show()
print("✅ Visualizaciones generadas")


## 💾 Paso 6: Guardar clusters en SQLite y reporte


In [ ]:
# PASO 6: Guardar en SQLite
print("=== GUARDANDO EN SQLITE ===\n")

df.to_sql('clientes_segmentados', conn, if_exists='replace', index=False)
conn.commit()

# Verificar
check = pd.read_sql_query(
    "SELECT cluster, cluster_nombre, COUNT(*) AS n_clientes FROM clientes_segmentados GROUP BY cluster",
    conn
)
print("Tabla 'clientes_segmentados' guardada:")
print(check.to_string(index=False))

print("\n📋 REPORTE FINAL DE SEGMENTACIÓN:")
print(f"  Algoritmo      : K-Means (scikit-learn)")
print(f"  k elegido      : {k_optimo}")
print(f"  Variables      : {features}")
print(f"  Inercia final  : {kmeans.inertia_:,.0f}")
print(f"  Silhouette     : {silhouette_score(X_scaled, kmeans.labels_):.4f}")
print("\n  Estrategia de negocio recomendada:")
for k_id, nombre in nombres_cluster.items():
    row = resumen.loc[k_id]
    print(f"  {nombre}")
    print(f"    → {int(row['n_clientes'])} clientes | Gasto prom: S/ {row['gasto_total_promedio']:.0f}")

conn.close()
print("\n🎉 ¡Clustering completado!")


---
## 💬 Actividad 3 – Análisis Reflexivo

**A)** ¿Cuál es la diferencia entre aprendizaje supervisado y no supervisado? ¿Por qué K-Means es no supervisado?

> _Tu respuesta aquí_

**B)** Elegiste k=X clusters. ¿Cómo lo justificas con el método del codo y el silhouette score?

> _Tu respuesta aquí_

**C)** Describe el perfil de cada cluster y qué estrategia de negocio aplicarías a cada uno.

> _Tu respuesta aquí_

**D)** ¿Por qué es importante escalar los datos antes de K-Means? Muestra un ejemplo numérico.

> _Tu respuesta aquí_


---
## ✅ Conclusiones
1. _Conclusión sobre el aprendizaje no supervisado y la segmentación_
2. _Conclusión sobre la selección de k con el método del codo_
3. _Conclusión sobre las decisiones de negocio basadas en clustering_


---
## 📚 Material Complementario
| Recurso | Enlace |
|---------|--------|
| Scikit-learn KMeans | https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html |
| StatQuest – K-Means | https://www.youtube.com/watch?v=4b5d3muPQmA |
| Elbow Method | https://towardsdatascience.com/elbow-method-k-means |
| Kaggle – Unsupervised Learning | https://www.kaggle.com/learn/unsupervised-learning |
